In [26]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Show full text in dataframe

# Load FinBERT
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

df = pd.read_csv("NIFTY50_news_2021_to_2026.csv")

df.head()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10365.60it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,ticker,title,date,source
0,ADANIPORTS.NS,Gujarat inks pact with Adani Ports for multi-modal logistics park - Construction Week India,25-01-2021,Construction Week India
1,ADANIPORTS.NS,Adani Ports signs MoU with Gujarat Govt - Indianchemicalnews.com,28-01-2021,Indianchemicalnews.com
2,ADANIPORTS.NS,Adani Ports receives clearance for expansion of Krishnapatnam Port - Ship Technology,04-01-2021,Ship Technology
3,ADANIPORTS.NS,Adani Group under scrutiny for links with brutal Myanmar military - Adani Watch,11-01-2021,Adani Watch
4,ADANIPORTS.NS,"Opposition takes on Adani's Rs 53,400-crore project ahead of public hearing - Business Standard",18-01-2021,Business Standard


In [28]:
def get_sentiment(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]

    return {
        "negative": probs[0].item(),
        "neutral": probs[1].item(),
        "positive": probs[2].item()
    }


In [29]:
df[['negative','neutral','positive']] = df['title'].apply(
    lambda x: pd.Series(get_sentiment(str(x)))
)
df['sentiment_score'] = df['positive'] - df['negative']

In [31]:
df

,ticker,title,date,source,negative,neutral,positive,sentiment_score
0,ADANIPORTS.NS,Gujarat inks pact with Adani Ports for multi-modal logistics park - Construction Week India,25-01-2021,Construction Week India,0.910678,0.015549,0.073773,-0.836905
1,ADANIPORTS.NS,Adani Ports signs MoU with Gujarat Govt - Indianchemicalnews.com,28-01-2021,Indianchemicalnews.com,0.082528,0.018252,0.899220,0.816691
2,ADANIPORTS.NS,Adani Ports receives clearance for expansion of Krishnapatnam Port - Ship Technology,04-01-2021,Ship Technology,0.906996,0.011264,0.081740,-0.825256
3,ADANIPORTS.NS,Adani Group under scrutiny for links with brutal Myanmar military - Adani Watch,11-01-2021,Adani Watch,0.023810,0.806574,0.169616,0.145806
4,ADANIPORTS.NS,"Opposition takes on Adani's Rs 53,400-crore project ahead of public hearing - Business Standard",18-01-2021,Business Standard,0.025240,0.045465,0.929295,0.904055
...,...,...,...,...,...,...,...,...
107287,ADANIPORTS.NS,"APSEZ, Gujarat sign MoU for 1,450-acre logistics park close to Sanand - Indian Transport & Logistics",25-01-2021,Indian Transport & Logistics,0.480553,0.014158,0.505289,0.024736
107288,ASIANPAINT.NS,Asian Paints looks back at a decade of SAP HANA deployment - Frontier Enterprise,13-01-2021,Frontier Enterprise,0.291660,0.024212,0.684127,0.392467
107289,ASIANPAINT.NS,The Asian Paints story - Finshots,29-01-2021,Finshots,0.034972,0.037932,0.927096,0.892125
107290,ASIANPAINT.NS,Asian Paints factory shut in Mysuru by farmers as the company goes back on job promise - The News Minute,13-01-2021,The News Minute,0.016126,0.917871,0.066003,0.049877


In [32]:
df['source'].value_counts()

source
The Economic Times                 5778
Business Standard                  4036
Moneycontrol.com                   3947
Mint                               3845
The Times of India                 3718
                                   ... 
Juneau Independent                 1   
Leo M. Bacha Funeral Home, Inc.    1   
Worcester Business Journal         1   
OregonLive.com                     1   
WTAJ                               1   
Name: count, Length: 5498, dtype: int64